# Genotype GISAID Sequences

Purpose: This notebook takes downloaded GISAID data and prepares it to be re-genotyped using multi-GenoFLU. All data is downloaded from GISAID on 8/24/2026, with release dates from 11/1/2021--8/21/2026, from the western hemisphere (Antarctica, North America, South America). The output is eight files, one per segment, containing all sequences from this time period.

## Housekeeping

In [1]:
import os
import pandas as pd
import dateutil
import shutil

In [2]:
# Paths

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 

# Collect user input

locations = "Antarctica,North America,South America"
# genotypes = ["A3"] 
start_date = "2021-11-01"
end_date = "2026-08-20"
date_range = dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y")

os.chdir(downloads)

# Create directories if needed
downloads_saved = home + "GISAID/downloads/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
gisaid_files = home + "GISAID/complete/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"

if not os.path.exists(gisaid_files): # checking if the directory exists or not
    os.makedirs(gisaid_files) # if the directory is not present then create it

# Get list of genotypes and states

os.chdir(references)

states = pd.read_csv("states_ref.csv")
animals_ref = pd.read_csv("animals_ref.csv")

## Configuring downloaded GISAID Files

In [7]:
def fasta_df(file_name, state_ref):
    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    segments = []
    collection_dates = []
    sequences = []
    # host_types = []
    species = []
    identifiers = []
    locations = []
    # genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    try:
                        split_header = header.split("|")
                        if len(header.split("|")) > 4:
                            identifier = header.split("|")[0]
                            identifiers.append(identifier)
                            split_first_header = split_header[1].split("/")
                        # else:
                        #     identifiers.append("unknown")
                        #     split_first_header = split_header[0].split("/")
                        # print(split_first_header)
                        # print(split_header)
                        headers.append(header) 
                        isolate_ids.append(split_first_header[-2]) # if "Catalonia" not in split_first_header[1] and "Navarra" not in split_first_header[1] else split_first_header[2])
                        locations.append(split_first_header[-3])
                        isolate_names.append(split_header[-4]) # We'll need to extract data from this too
                        # print(split_header[2].split("_")[-1])
                        subtypes.append(split_header[-3].split("_")[-1])  # Get only H5N1
                        # genotypes.append(split_header[-1])
                        segments.append(split_header[-2]) # .split("|")[-1])
                        # host_types.append(split_header[-2])
                        species.append(split_first_header[1])
                        # if split_header[4] == "2024-01-01":
                        #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                        # elif split_header[4] == "2025-01-01":
                        #     collection_dates.append("2025")
                        # else: 
                        collection_dates.append(split_header[-1])
                        # collection_dates.append(split_header[-1].split("_")[-1])
                        if num < len(lines): # If we're not at the last line
                            # for i, l in enumerate(lines[num + 1:]):
                            i = num
                            sequence = ""
                            # print(lines[i])
                            # print(lines[i + 1])
                            while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                                sequence = sequence + lines[i + 1].strip()
                                i += 1
                            sequences.append(sequence) # Add next line to sequences
                    except:
                        headers.remove(header)
                        identifiers.remove(identifier)
                        print(header)
                        continue
        f.close()

    
    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    fasta["Segment"] = segments
    fasta["Location_Header"] = locations
    # Geo_Location is more complicated
    try:
        fasta["Geo_Location"] = fasta["Location_Header"].apply( # lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))

                                                    lambda x: 
                                                    # If "x" has the state abbreviation (e.g. "MD")
                                                    state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                    + "-" + 
                                                    x.split(" ")[-1]
                                                    if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                    # If "x" has the full state name (e.g. "Maryland")
                                                    else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                    + "-" + 
                                                    state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                    if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                    # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                    else 
                                                    x
                                                    )
        fasta["Geo_Location"] = fasta["Geo_Location"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)
    except:
        print("Geo Location not found.")
        fasta["Geo_Location"] = fasta["Location_Header"]
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    # fasta["Host_Type"] = host_types
    # fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    # if len(identifiers) == len(fasta):
    fasta["Identifier"] = identifiers
    fasta = fasta[~fasta["Date Collected"].str.contains("placed_under_publishing_embargo")]
    fasta["Date Collected"] = fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x)
    
        
    return fasta

# Function to normalize hosts
# Fix animals in host type

def fix_animals(fasta, animals_ref):

    # If the animal is in a specific column of animals_ref, label host type as column name
    animal_list = [] # Find animals first
    for name in fasta["Isolate_Name"].values:
        try:
            animal = name.split("/")[1]
            animal_low = animal.lower()
        except:
            animal_low = "unknown"
        animal_list.append(animal_low)


    animal_types = []
    for animal in animal_list: # Label each animal as a type
        # print(animal)
        if animal in animals_ref["wild_avian"].values:
            animal_types.append("wild_avian")
        elif animal in animals_ref["domestic_avian"].values:
            animal_types.append("domestic_avian")
        elif animal in animals_ref["cattle"].values:
            animal_types.append("cattle")
        elif animal in animals_ref["feline"].values:
            animal_types.append("feline")
        elif animal in animals_ref["other_mammal"].values:
            animal_types.append("other_mammal")
        elif animal in animals_ref["human"].values:
            animal_types.append("human")
        elif animal in animals_ref["pet_food"].values:
            animal_types.append("pet_food")
        elif animal in animals_ref["experimental"].values:
            animal_types.append("experimental")
        else: # If other
            animal_types.append("other")

    fasta["Host_Type"] = animal_types

    return fasta

# Function to get each unique animal listed so we can sort them

def sort_animals(fasta):
    isolate_names = fasta["Isolate_Name"]
    animal_list = []
    for name in isolate_names.values:
        # print(name)
        try:
            animal = name.split("/")[1]
            animal_low = animal.lower()
            animal_list.append(animal_low)
        except:
            continue

    unique_animals = list(set(animal_list))

    # Save the animals to a file so we can sort them
    return unique_animals

# Function to get metadata
def separate_fasta_by_segments(metadata, fasta, animals_df): #, b313_fasta, d11_fasta):

    fasta = fix_animals(fasta, animals_df) # Fix animals first

    unique_segments = list(set(fasta["Segment"])) # Get list of segments

    # “>EPI_ID|Isolate_name|subtype|collection_date|host_type|genotype”

    segment_fastas = [] # Get a list of fastas, separated by segment
    for seg in unique_segments: # For each segment
        xls = metadata[metadata["Publishing_Embargo_Until"].isna()] # [metadata["Genotype"].apply(lambda x: x.split(" ")[0]) == genotype] # Get only the metadata corresponding to that genotype
        xls = xls.rename(columns={"Isolate_Id":"Identifier"})

        fasta_seg_pre = fasta.merge(xls, how="right", on="Identifier")

        fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

        # Rename sequences 
        new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name_x"] + "|" + fasta_seg["Subtype_x"] + "|" + fasta_seg["Geo_Location"] + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_seg["Host_Type"] # + "|" + fasta_seg["Genotype"] #.apply(lambda x: "" if x != "human" else "|human")
        fasta_seg["full_header"] = new_name
        fasta_seg = fasta_seg.rename(columns={"Isolate_Name_x":"Isolate_Name"})
        # print(fasta_seg["New_Name"])

        segment_fastas.append(fasta_seg)
        print(fasta_seg[["full_header", "Header", "Isolate_Id", "Isolate_Name", "Subtype_x", "Segment", "Geo_Location", "Date Collected", "Identifier", "Host_Type", "Isolate_Name_y", "Subtype_y", "Genotype", "Location", "Collection_Date"]])


    return segment_fastas, unique_segments

In [8]:
all_metadata_files = []
all_fasta_files = []

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads):
    if len(files) > 0: # If we have any files that need to be moved
        for file in files:
            file_name = os.path.join(dirpath, file)
            destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
            try:
                shutil.move(file_name, destination_path)
            except:
                print("Error moving file", file_name)
                continue 
    else: # If we have downloaded files saved already
        continue
    break 

for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)

        print(file_name)

        # Now go through files and get contents
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name)
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)
    break 

C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-08-20_Antarctica_North_America_South_America/gisaid_epiflu_isolates (1).xls
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-08-20_Antarctica_North_America_South_America/gisaid_epiflu_isolates (2).xls
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-08-20_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-08-20_Antarctica_North_America_South_America/gisaid_epiflu_sequence (1).fasta
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/

In [9]:
# Concatenate metadata

metadata_concat = pd.DataFrame()
for metadata_file in all_metadata_files:
    metadata_concat = pd.concat([metadata_concat, metadata_file])

In [10]:
# Separate fastas by segment -- results in number of downloaded fastas * number of genotypes * 8 segments
segment_fastas = []
unique_animals_all = []
for i, fasta in enumerate(all_fasta_files):

    metadata = metadata_concat

    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    os.chdir(references)
    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_segments(metadata, fasta, animals_ref) # Separate the fasta dataframes into 8 different files based on segment

    for fasta in fastas:
        segment_fastas.append(fasta)

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_32292\3047170977.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["full_header"] = new_name


                                             full_header  \
5      >EPI_ISL_19660815|A/chicken/Utah/22-020377-010...   
13     >EPI_ISL_19660816|A/chicken/Washington/22-0203...   
21     >EPI_ISL_19660897|A/trumpeter_swan/Wyoming/23-...   
29     >EPI_ISL_19660900|A/red-tailed_hawk/Wyoming/23...   
37     >EPI_ISL_19660918|A/peregrine_falcon/Minnesota...   
...                                                  ...   
79181  >EPI_ISL_19593808|A/dairy_cow/California/03399...   
79189  >EPI_ISL_19593805|A/dairy_cow/California/03399...   
79193  >EPI_ISL_19660248|A/California/216/2024|H5N1|U...   
79201  >EPI_ISL_19628008|A/California/213/2024|H5N1|U...   
79209  >EPI_ISL_19726293|A/Nevada/10/2025|H5N1|USA-NV...   

                                                  Header  \
5      EPI_ISL_19660815|A/chicken/Utah/22-020377-010-...   
13     EPI_ISL_19660816|A/chicken/Washington/22-02039...   
21     EPI_ISL_19660897|A/trumpeter_swan/Wyoming/23-0...   
29     EPI_ISL_19660900|A/red-tailed_ha

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_32292\3047170977.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["full_header"] = new_name


                                             full_header  \
2      >EPI_ISL_19660815|A/chicken/Utah/22-020377-010...   
10     >EPI_ISL_19660816|A/chicken/Washington/22-0203...   
18     >EPI_ISL_19660897|A/trumpeter_swan/Wyoming/23-...   
26     >EPI_ISL_19660900|A/red-tailed_hawk/Wyoming/23...   
34     >EPI_ISL_19660918|A/peregrine_falcon/Minnesota...   
...                                                  ...   
79178  >EPI_ISL_19593808|A/dairy_cow/California/03399...   
79186  >EPI_ISL_19593805|A/dairy_cow/California/03399...   
79199  >EPI_ISL_19660248|A/California/216/2024|H5N1|U...   
79207  >EPI_ISL_19628008|A/California/213/2024|H5N1|U...   
79215  >EPI_ISL_19726293|A/Nevada/10/2025|H5N1|USA-NV...   

                                                  Header  \
2      EPI_ISL_19660815|A/chicken/Utah/22-020377-010-...   
10     EPI_ISL_19660816|A/chicken/Washington/22-02039...   
18     EPI_ISL_19660897|A/trumpeter_swan/Wyoming/23-0...   
26     EPI_ISL_19660900|A/red-tailed_ha

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_32292\3047170977.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["full_header"] = new_name


...

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_32292\3047170977.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["full_header"] = new_name


                                             full_header  \
15940  >EPI_ISL_19186563|A/dairy_cow/Michigan/24-0103...   
15948  >EPI_ISL_19466158|A/South_American_tern/Argent...   
15956  >EPI_ISL_19466210|A/South_American_tern/Argent...   
15964  >EPI_ISL_19466183|A/southern_elephant_seal/Arg...   
15972  >EPI_ISL_19466181|A/royal_tern/Argentina/CH-PD...   
...                                                  ...   
85572  >EPI_ISL_19152880|A/Great_Horned_Owl/AB/FAV-08...   
85580  >EPI_ISL_19070498|A/pinniped/Uruguay/P4_6923/2...   
85588  >EPI_ISL_19152518|A/Northern_Gannet/QC/FAV-061...   
85596  >EPI_ISL_19152002|A/Wood_Duck/ON/FAV-1310-2/20...   
85604  >EPI_ISL_19152700|A/American_Crow/QC/FAV-0870-...   

                                                  Header         Isolate_Id  \
15940  EPI_ISL_19186563|A/dairy_cow/Michigan/24-01030...  24-010303-001-300   
15948  EPI_ISL_19466158|A/South_American_tern/Argenti...           CH-PD030   
15956  EPI_ISL_19466210|A/South_American_t

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_32292\3047170977.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["full_header"] = new_name


                                             full_header  \
15945  >EPI_ISL_19186563|A/dairy_cow/Michigan/24-0103...   
15953  >EPI_ISL_19466158|A/South_American_tern/Argent...   
15961  >EPI_ISL_19466210|A/South_American_tern/Argent...   
15969  >EPI_ISL_19466183|A/southern_elephant_seal/Arg...   
15977  >EPI_ISL_19466181|A/royal_tern/Argentina/CH-PD...   
...                                                  ...   
85577  >EPI_ISL_19152880|A/Great_Horned_Owl/AB/FAV-08...   
85585  >EPI_ISL_19070498|A/pinniped/Uruguay/P4_6923/2...   
85593  >EPI_ISL_19152518|A/Northern_Gannet/QC/FAV-061...   
85601  >EPI_ISL_19152002|A/Wood_Duck/ON/FAV-1310-2/20...   
85609  >EPI_ISL_19152700|A/American_Crow/QC/FAV-0870-...   

                                                  Header         Isolate_Id  \
15945  EPI_ISL_19186563|A/dairy_cow/Michigan/24-01030...  24-010303-001-300   
15953  EPI_ISL_19466158|A/South_American_tern/Argenti...           CH-PD030   
15961  EPI_ISL_19466210|A/South_American_t

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_32292\3047170977.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["full_header"] = new_name


## Make FASTA files for GenoFLU-multi

In [17]:
def df_to_fasta(fasta, file_name, output_path):

    output_file = open(output_path + file_name, "a")

    for index, row in fasta.iterrows():
        name = fasta.loc[index, "full_header"]
        sequence = fasta.loc[index, "sequence"]
    # First is header, second is sequence
        output_file.write(name + "\n")
        output_file.write(sequence + "\n")
    output_file.close()

In [18]:
os.chdir(gisaid_files)

for segment_fasta in segment_fastas:
    segment = segment_fasta["Segment"].values[0]
    # print(segment_fasta)
    segment_fasta["sequence"] = segment_fasta["Sequence"]
    header_sequence = segment_fasta[["full_header", "Sequence"]]
    df_to_fasta(segment_fasta, segment + "_segment_all_" + date_range + ".fasta", gisaid_files)


## Modify results to include original genotypes

In [19]:
# It would be nice to see original genotypes prior to new genotypes

genoflu_results = pd.read_csv("results.tsv", delimiter="\t")
genoflu_results["Identifier"] = genoflu_results["Strain"].apply(lambda x: x.split("|")[0])

og_genotypes = genoflu_results.merge(segment_fastas[0], on="Identifier")


In [21]:
og_genotypes = og_genotypes.rename(columns={"Genotype_x":"genotype_new", "Genotype_y":"genotype_old"})

In [25]:
og_genotypes["genotype_old_clean"] = og_genotypes["genotype_old"].apply(lambda x: x.split(" (")[0])
og_genotypes[["Strain", "genotype_new", "genotype_old_clean", "genotype_old", "Identifier"]]

,Strain,genotype_new,genotype_old_clean,genotype_old,Identifier
0,EPI_ISL_19660815|A/chicken/Utah/22-020377-010-...,B3.2,B3.2,B3.2 (<i style='font-size:11px'>GenoFLU</i>) /...,EPI_ISL_19660815
1,EPI_ISL_19660816|A/chicken/Washington/22-02039...,B4.1,B4.1,B4.1 (<i style='font-size:11px'>GenoFLU</i>) /...,EPI_ISL_19660816
2,EPI_ISL_19660897|A/trumpeter_swan/Wyoming/23-0...,B3.6,B3.6,B3.6 (<i style='font-size:11px'>GenoFLU</i>) /...,EPI_ISL_19660897
3,EPI_ISL_19660900|A/red-tailed_hawk/Wyoming/23-...,B3.7,B3.7,B3.7 (<i style='font-size:11px'>GenoFLU</i>) /...,EPI_ISL_19660900
4,EPI_ISL_19660918|A/peregrine_falcon/Minnesota/...,B3.6,B3.6,B3.6 (<i style='font-size:11px'>GenoFLU</i>) /...,EPI_ISL_19660918
...,...,...,...,...,...
9897,EPI_ISL_19593808|A/dairy_cow/California/033991...,B3.13,B3.13,B3.13 (<i style='font-size:11px'>GenoFLU</i>) ...,EPI_ISL_19593808
9898,EPI_ISL_19593805|A/dairy_cow/California/033991...,B3.13,B3.13,B3.13 (<i style='font-size:11px'>GenoFLU</i>) ...,EPI_ISL_19593805
9899,EPI_ISL_19660248|A/California/216/2024|H5N1|US...,B3.13,B3.13,B3.13 (<i style='font-size:11px'>GenoFLU</i>) ...,EPI_ISL_19660248
9900,EPI_ISL_19628008|A/California/213/2024|H5N1|US...,B3.13,B3.13,B3.13 (<i style='font-size:11px'>GenoFLU</i>) ...,EPI_ISL_19628008


In [27]:
# Save
og_genotypes[["Strain", "genotype_new", "genotype_old_clean", "genotype_old", "Identifier"]].to_csv("gisaid_genoflu_retyping_results.csv", index=None)

In [28]:
len(segment_fastas)

24